# 6.1b 传统机器学习与软投票

基于 CTMP 清洗数据集（6 类、24 kHz、5 秒切片），用 98 维手工特征 + sklearn Pipeline
训练三个分类器（SVM、Random Forest、XGBoost）及 RF+XGB 软投票融合。

评估协议：
- **内部**：train split 训练 → test split 评估
- **外部**：同一模型直接在 external_test（ChMusic）上推理
- **划分敏感性**：在 3 份冻结划分上评估，取均值和总体标准差（`ddof=0`）；external_test固定不变

> 数据来源：`_data_pipeline/output/`。若未跑过清洗流程，请先按
> `06_1a_dataset_exploration.ipynb` §9.0 的步骤生成。


## 1. 环境准备


In [ ]:
import sys
from pathlib import Path

# 路径推断：从 cwd 向上找含 CODE/datasets 的目录；PROJECT_ROOT 指向 CODE/
_p = Path.cwd()
while not (_p / "CODE" / "datasets").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/datasets 的目录），请在项目内运行本 Notebook")
    _p = _parent
PROJECT_ROOT = _p / "CODE"  # CODE/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT:', PROJECT_ROOT)


### 1.1 依赖与数据齐备性检查

下面的单元统一检查本 Notebook 所需的 Python 包和 CTMP 清洗输出。
若依赖缺失，检查会立即报错并列出相应的安装命令。


In [ ]:
from chapter06._common import check_environment

check_environment(notebook='06_1b', require_ctmp=True)


In [ ]:
import subprocess
import tempfile
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from chapter06._common import (
    add_recall_colorbar,
    plot_confusion_matrix,
    setup_chinese_font,
)
from chapter06._common.ctmp_loader import (
    CTMP_CLASSES,
    build_label_map,
    get_ctmp_output_dir,
    load_ctmp_segments,
)
from chapter06.traditional_ml.features import FEATURE_DIM, extract_features

setup_chinese_font()

OUT_DIR = PROJECT_ROOT / 'chapter06' / 'traditional_ml' / 'outputs'
FIG_DIR = OUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [0, 1, 2]
# 0/1/2 是三份冻结 manifest 的接口编号；同一数值也传给分类器 random_state。
LABEL_MAP = build_label_map()
CLASS_NAMES = list(CTMP_CLASSES)
N_CLASSES = len(CLASS_NAMES)
print(f'CTMP 类别: {CLASS_NAMES} ({N_CLASSES} 类)')
print(f'CTMP 输出目录: {get_ctmp_output_dir()}')


## 2. 加载数据与特征提取

CTMP 切片已是 24 kHz、5 秒 wav 文件，直接 `soundfile.read` 后提取 98 维特征：

- MFCC mean/std（26 维）+ Delta-MFCC mean/std（26 维）
- Chroma mean/std（24 维）
- Spectral Contrast mean/std（14 维）
- 频谱质心、过零率、RMS、Onset Strength 各 mean/std（8 维）

为 3 份冻结划分分别构建 train / test 特征矩阵。三份清单中的 external_test
音频逐项一致，因此只提取一次并在三份评估中复用。


In [ ]:
def load_features(seed: int, split: str) -> tuple[np.ndarray, np.ndarray]:
    """加载指定冻结划分编号 × split 的音频并提取特征。"""
    segments = load_ctmp_segments(seed=seed, split=split)
    X = np.empty((len(segments), FEATURE_DIM), dtype=np.float32)
    y = np.empty(len(segments), dtype=np.int64)
    for i, seg in enumerate(segments):
        audio, sr = sf.read(seg['audio_path'], dtype='float32')
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        X[i] = extract_features(audio, sr=sr)
        y[i] = LABEL_MAP[seg['family_label']]
    return X, y

t0 = time.perf_counter()
data = {}
for seed in SEEDS:
    data[seed] = {}
    for split in ('train', 'test'):
        X, y = load_features(seed, split)
        data[seed][split] = (X, y)
        print(f'  split {seed} / {split:14s}: X={X.shape}, y 分布={np.bincount(y, minlength=N_CLASSES).tolist()}')

external_signatures = []
for seed in SEEDS:
    segments = load_ctmp_segments(seed=seed, split='external_test')
    external_signatures.append([
        (seg['segment_id'], seg['family_label'], seg['sample_id']) for seg in segments
    ])
if any(sig != external_signatures[0] for sig in external_signatures[1:]):
    raise RuntimeError('三份冻结清单的 external_test 不一致，不能共用特征矩阵')

external_features = load_features(SEEDS[0], 'external_test')
for seed in SEEDS:
    data[seed]['external_test'] = external_features
X_ext, y_ext = external_features
print(f'  共用 external_test: X={X_ext.shape}, y 分布={np.bincount(y_ext, minlength=N_CLASSES).tolist()}')
elapsed = time.perf_counter() - t0
print(f'\n特征提取完毕，总用时 {elapsed:.1f}s')


## 3. 分类器定义

每个分类器包在 `Pipeline(StandardScaler, clf)` 里——StandardScaler 仅在训练集上 fit，
不会让测试集统计信息泄露。

训练片段的类别数不完全相等；例如冻结划分0中笙为245段、琵琶为65段。代码使用
`class_weight='balanced'` 或等价的逐样本权重，按训练集类别频率调整损失贡献：

- `SVC(class_weight='balanced', probability=True)`：RBF核；内部另拟合概率校准参数。
- `RandomForestClassifier(class_weight='balanced')`：集成基于自助采样训练的多棵决策树，以降低模型方差。
- `XGBClassifier`：不支持 `class_weight`，通过 `sample_weight` 等效实现。
- 软投票仅含RF与XGBoost。若把SVM概率一并平均，应先在验证集检查各模型的校准质量，
  不能仅因都输出0到1之间的数值就假定概率可直接等权比较。


In [ ]:
def make_models(seed: int) -> dict[str, Pipeline]:
    def _pipe(clf):
        return Pipeline([('scaler', StandardScaler()), ('clf', clf)])
    return {
        'SVM': _pipe(SVC(C=1.0, kernel='rbf', probability=True,
                         class_weight='balanced', random_state=seed)),
        'RF': _pipe(RandomForestClassifier(
            n_estimators=300, class_weight='balanced',
            random_state=seed, n_jobs=1)),
    }


def run_xgb_worker(
    seed: int,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    X_ext: np.ndarray,
    sample_weight: np.ndarray,
) -> dict[str, np.ndarray]:
    """在独立进程中拟合并评估 XGBoost，返回预测与特征重要性。"""
    worker = PROJECT_ROOT / 'chapter06' / 'traditional_ml' / 'xgb_worker.py'
    with tempfile.TemporaryDirectory(prefix='chapter06_xgb_') as tmp_dir:
        input_path = Path(tmp_dir) / 'input.joblib'
        output_path = Path(tmp_dir) / 'output.joblib'
        joblib.dump({
            'seed': seed,
            'X_train': X_train,
            'y_train': y_train,
            'X_test': X_test,
            'X_ext': X_ext,
            'sample_weight': sample_weight,
        }, input_path)
        subprocess.run(
            [sys.executable, str(worker), str(input_path), str(output_path)],
            check=True,
        )
        return joblib.load(output_path)

DISPLAY_NAMES = {
    'SVM': 'SVM 基线',
    'RF': '随机森林',
    'XGB': 'XGBoost',
    'Voting': '软投票（RF + XGB）',
}


## 4. 训练与评估（三份冻结划分 × 4种方法）

每份冻结划分：在 train 上 fit → 在 test（内部）和 external_test（外部）上分别评估。
代码接口仍使用 `seed=0/1/2`，且把同一数值传给分类器的 `random_state`；因此三行结果
对应三份数据划分与各自固定的分类器随机状态，不是同一划分上的三次随机重复。
软投票直接平均当次已经拟合且使用了相同类别平衡协议的 RF 与 XGBoost
概率，不重新克隆或拟合基模型。


In [ ]:
from sklearn.utils.class_weight import compute_sample_weight

all_results = []
seed0_models = None
seed0_xgb_importance = None

for seed in SEEDS:
    X_train, y_train = data[seed]['train']
    X_test, y_test = data[seed]['test']
    X_ext, y_ext = data[seed]['external_test']

    # XGBoost 不支持 class_weight 参数，用 sample_weight 等效实现
    sw = compute_sample_weight('balanced', y_train)

    models = make_models(seed)
    # SVM 与随机森林在当前进程中拟合。
    for name, pipe in models.items():
        print(f'split {seed} / {name}: 开始训练', flush=True)
        pipe.fit(X_train, y_train)
        print(f'split {seed} / {name}: 训练完成', flush=True)

    if seed == SEEDS[0]:
        seed0_models = models

    for name, pipe in models.items():
        pred_test = pipe.predict(X_test)
        pred_ext = pipe.predict(X_ext)
        all_results.append({
            'seed': seed,
            'model': name,
            'test_acc': accuracy_score(y_test, pred_test),
            'test_f1': f1_score(y_test, pred_test, average='macro'),
            'ext_acc': accuracy_score(y_ext, pred_ext),
            'ext_f1': f1_score(y_ext, pred_ext, average='macro'),
            'pred_test': pred_test,
            'pred_ext': pred_ext,
            'y_test': y_test,
            'y_ext': y_ext,
        })

    # 主进程不创建 XGBoost 对象。子进程使用相同的 StandardScaler、模型参数
    # 和类别平衡样本权重，并只返回普通数组。
    print(f'split {seed} / XGB: 开始训练', flush=True)
    xgb_output = run_xgb_worker(seed, X_train, y_train, X_test, X_ext, sw)
    print(f'split {seed} / XGB: 训练完成', flush=True)
    pred_test = xgb_output['pred_test']
    pred_ext = xgb_output['pred_ext']
    all_results.append({
        'seed': seed,
        'model': 'XGB',
        'test_acc': accuracy_score(y_test, pred_test),
        'test_f1': f1_score(y_test, pred_test, average='macro'),
        'ext_acc': accuracy_score(y_ext, pred_ext),
        'ext_f1': f1_score(y_ext, pred_ext, average='macro'),
        'pred_test': pred_test,
        'pred_ext': pred_ext,
        'y_test': y_test,
        'y_ext': y_ext,
    })
    if seed == SEEDS[0]:
        seed0_xgb_importance = xgb_output['feature_importances']

    # 直接复用已经拟合的 RF 概率与子进程返回的 XGBoost 概率。
    proba_test = 0.5 * (
        models['RF'].predict_proba(X_test) + xgb_output['proba_test']
    )
    proba_ext = 0.5 * (
        models['RF'].predict_proba(X_ext) + xgb_output['proba_ext']
    )
    pred_test = proba_test.argmax(axis=1)
    pred_ext = proba_ext.argmax(axis=1)
    all_results.append({
        'seed': seed,
        'model': 'Voting',
        'test_acc': accuracy_score(y_test, pred_test),
        'test_f1': f1_score(y_test, pred_test, average='macro'),
        'ext_acc': accuracy_score(y_ext, pred_ext),
        'ext_f1': f1_score(y_ext, pred_ext, average='macro'),
        'pred_test': pred_test,
        'pred_ext': pred_ext,
        'y_test': y_test,
        'y_ext': y_ext,
    })
    # 后续只需划分 0 的树模型绘制特征重要性；及时释放其余划分的模型，避免多组大规模树集成与高分辨率图像同时驻留内存。
    del models, xgb_output
    pipe = None  # pipe 为循环残留变量，置 None 释放最后一个 pipeline 的引用
    print(f'split {seed} 完成')

df_results = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ('pred_test', 'pred_ext', 'y_test', 'y_ext')}
    for r in all_results
])
print(df_results.to_string(index=False))


## 5. 主结果表：3 份冻结划分的均值 ± 总体标准差

内部（test）与外部（external_test）的准确率和 macro-F1 并列，Gap = 内部 − 外部。
标准差按总体标准差计算（`ddof=0`）。


In [ ]:
summary_rows = []
for model_name in ['SVM', 'RF', 'XGB', 'Voting']:
    sub = df_results[df_results['model'] == model_name]
    summary_rows.append({
        '方法': DISPLAY_NAMES[model_name],
        '内部 Acc': f"{sub['test_acc'].mean():.3f} ± {sub['test_acc'].std(ddof=0):.3f}",
        '内部 F1': f"{sub['test_f1'].mean():.3f} ± {sub['test_f1'].std(ddof=0):.3f}",
        '外部 Acc': f"{sub['ext_acc'].mean():.3f} ± {sub['ext_acc'].std(ddof=0):.3f}",
        '外部 F1': f"{sub['ext_f1'].mean():.3f} ± {sub['ext_f1'].std(ddof=0):.3f}",
        'Gap (Acc)': f"{sub['test_acc'].mean() - sub['ext_acc'].mean():+.3f}",
    })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(OUT_DIR / 'method_comparison.csv', index=False)
print('written:', (OUT_DIR / 'method_comparison.csv').resolve())
df_summary


## 6. 混淆矩阵：内部 vs 外部（划分 0，RF）

固定展示随机森林在划分 0 上的结果。左图为 test（内部），右图为
external_test（外部）；不根据这两个测试集的分数选择展示对象。


In [ ]:
# 找 seed 0 中各模型的结果
seed0_results = {r['model']: r for r in all_results if r['seed'] == 0}

# 固定展示 RF，避免根据 test 或 external_test 分数选择可视化对象
best_name = 'RF'
r = seed0_results[best_name]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_confusion_matrix(
    axes[0], r['y_test'], r['pred_test'],
    class_names=CLASS_NAMES, title=f'{DISPLAY_NAMES[best_name]} — 内部 test',
    labels=list(range(N_CLASSES)),
)
plot_confusion_matrix(
    axes[1], r['y_ext'], r['pred_ext'],
    class_names=CLASS_NAMES, title=f'{DISPLAY_NAMES[best_name]} — 外部 external_test',
    labels=list(range(N_CLASSES)),
)
fig.subplots_adjust(wspace=0.4)
add_recall_colorbar(fig, axes)
fig.savefig(FIG_DIR / 'confusion_internal_vs_external.png', dpi=600, bbox_inches='tight')
plt.show()
plt.close(fig)


### 6.1 四模型外部混淆矩阵并排

把4个模型在external_test上的混淆矩阵并排，比较不同方法在外部数据上的误判分布。


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, model_name in zip(axes, ['SVM', 'RF', 'XGB', 'Voting']):
    r = seed0_results[model_name]
    plot_confusion_matrix(
        ax, r['y_ext'], r['pred_ext'],
        class_names=CLASS_NAMES, title=DISPLAY_NAMES[model_name],
        labels=list(range(N_CLASSES)),
    )
fig.subplots_adjust(wspace=0.35)
add_recall_colorbar(fig, axes)
fig.savefig(FIG_DIR / 'confusion_external_all_models.png', dpi=600, bbox_inches='tight')
plt.show()
plt.close(fig)


## 7. 特征重要性（随机森林 / XGBoost）

98维特征中，树模型实际使用了哪些？取冻结划分0的已训练模型，展示top-30。

观察要点：模型是否利用了 Delta-MFCC / Spectral Contrast 等新增特征，
还是仍然集中在 MFCC 均值等绝对量上。


In [ ]:
feat_names = (
    [f'MFCC{i}_均值' for i in range(13)]
    + [f'MFCC{i}_标准差' for i in range(13)]
    + [f'ΔMFCC{i}_均值' for i in range(13)]
    + [f'ΔMFCC{i}_标准差' for i in range(13)]
    + [f'Chroma{i}_均值' for i in range(12)]
    + [f'Chroma{i}_标准差' for i in range(12)]
    + [f'Contrast{i}_均值' for i in range(7)]
    + [f'Contrast{i}_标准差' for i in range(7)]
    + ['频谱质心_均值', '频谱质心_标准差', '过零率_均值', '过零率_标准差',
       'RMS_均值', 'RMS_标准差', 'Onset_均值', 'Onset_标准差']
)
group_color = (
    ['0.20'] * 13 + ['0.35'] * 13
    + ['0.15'] * 13 + ['0.30'] * 13
    + ['0.55'] * 12 + ['0.70'] * 12
    + ['0.45'] * 7 + ['0.60'] * 7
    + ['0.25'] * 8
)

# 取 seed 0 的 RF 与 XGBoost 特征重要性
pipe_rf = seed0_models['RF']
imp_rf = pipe_rf.named_steps['clf'].feature_importances_
imp_xgb = seed0_xgb_importance

TOP_K = 30
fig, axes = plt.subplots(1, 2, figsize=(14, 8), sharey=True)
order = np.argsort(imp_rf)[::-1][:TOP_K]
axes[0].barh(np.array(feat_names)[order][::-1], imp_rf[order][::-1],
             color=np.array(group_color)[order][::-1])
axes[0].set_title('随机森林 特征重要性 (top-30)')
axes[0].set_xlabel('重要性')

order = np.argsort(imp_xgb)[::-1][:TOP_K]
axes[1].barh(np.array(feat_names)[order][::-1], imp_xgb[order][::-1],
             color=np.array(group_color)[order][::-1])
axes[1].set_title('XGBoost 特征重要性 (top-30)')
axes[1].set_xlabel('重要性')

for ax in axes:
    ax.tick_params(axis='y', labelsize=7)
fig.tight_layout()
fig.savefig(FIG_DIR / 'feature_importance.png', dpi=600, bbox_inches='tight')
plt.show()
plt.close(fig)
